In [0]:
# ==============================================================================
# CONFIGURAÇÕES
# ==============================================================================

import json
import pandas as pd
import pyarrow as pa
from datetime import datetime, timezone
from deltalake import DeltaTable
from deltalake.writer import write_deltalake

# Constantes locais
CONTAINER = "squad1"

def get_delta_path(camada: str, tabela: str, storage_opts: dict) -> str:
    """Usa a URI nativa abfss:// com tratamento para gravação na raiz (camada vazia)."""
    conta = storage_opts.get("account_name")
    
    # Se a camada estiver vazia, grava na raiz sem colocar barras duplas
    if not camada:
        return f"abfss://{CONTAINER}@{conta}.dfs.core.windows.net/{tabela}"
        
    return f"abfss://{CONTAINER}@{conta}.dfs.core.windows.net/{camada}/{tabela}"

def delta_existe(camada: str, tabela: str, storage_opts: dict) -> bool:
    try:
        DeltaTable(get_delta_path(camada, tabela, storage_opts), storage_options=storage_opts)
        return True
    except Exception:
        return False

def gravar_delta(df, camada: str, tabela: str, storage_opts: dict, mode: str = "append", particionar: bool = True) -> bool:
    path = get_delta_path(camada, tabela, storage_opts)
    modo_real = mode if delta_existe(camada, tabela, storage_opts) else "overwrite"

    try:
        # 1. Conversão para Pandas
        pdf = df.toPandas()

        # 2. Correção OBRIGATÓRIA de fuso horário (Evita erro fatal no PyArrow)
        for col_name in pdf.columns:
            if pd.api.types.is_datetime64_any_dtype(pdf[col_name]):
                pdf[col_name] = pdf[col_name].dt.tz_localize(None)

        # 3. Conversão para Tabela Arrow
        tabela_arrow = pa.Table.from_pandas(pdf, preserve_index=False)

        # 4. Definição Dinâmica de Partições
        partition_by = None
        if particionar and camada == "bronze":
            # Nomes das partições que você configurou
            possiveis = ["bronze_ingest_year", "bronze_ingest_month", "bronze_ingest_day", "bronze_ingest_hour"]
            colunas_pdf = pdf.columns.tolist()
            # Pega apenas as partições que realmente existem no DataFrame
            partition_by = [c for c in possiveis if c in colunas_pdf]
            if not partition_by: 
                partition_by = None

        # 5. Gravação via SDK (Bypass do Databricks Serverless)
        write_deltalake(
            table_or_uri=path,
            data=tabela_arrow,
            mode=modo_real,
            storage_options=storage_opts,
            partition_by=partition_by,
            # Se for overwrite, sobrescreve o schema. Se for append, permite merge.
            schema_mode="overwrite" if modo_real == "overwrite" else "merge"
        )
        
        print(f"[Sucesso] Gravado fisicamente em: {path} | Linhas: {len(pdf)}")
        return True

    except Exception as e:
        print(f"[Erro] Falha ao gravar {path}: {str(e)}")
        return False

In [0]:
# ==============================================================================
# CAMADA BRONZE
# ==============================================================================
import json
from datetime import datetime, timezone

# ==============================================================================
# FUNÇÕES AUXILIARES DE ADLS E CONTROLE (JSON) 
# ==============================================================================

def garantir_diretorio(container_client, caminho):
    """Cria diretório no ADLS se ainda não existir."""
    try:
        container_client.create_directory(caminho)
        print(f"Diretório criado: {caminho}")
    except Exception:
        pass # Diretório já existe

def ler_json_adls(container_client, caminho_arquivo, padrao=None):
    """Lê JSON no ADLS. Se não existir, retorna o padrão."""
    if padrao is None:
        padrao = []
    try:
        file_client = container_client.get_file_client(caminho_arquivo)
        conteudo = file_client.download_file().readall().decode("utf-8")
        if not conteudo.strip():
            return padrao
        return json.loads(conteudo)
    except Exception:
        return padrao

def salvar_json_adls(container_client, caminho_arquivo, dados):
    """Salva JSON no ADLS, criando diretórios pais se necessário."""
    pasta_pai = "/".join(caminho_arquivo.split("/")[:-1])
    garantir_diretorio(container_client, pasta_pai)

    file_client = container_client.get_file_client(caminho_arquivo)
    conteudo = json.dumps(dados, ensure_ascii=False, indent=2)
    file_client.upload_data(conteudo, overwrite=True)
    print(f"JSON de controle salvo em: {caminho_arquivo}")

def listar_controles_entidade(container_client, pasta_controle_base):
    """Lista todos os arquivos controle_leitura.json de uma entidade específica."""
    controles = []
    try:
        for item in container_client.get_paths(path=pasta_controle_base, recursive=True):
            if item.name.endswith("controle_leitura.json"):
                controles.append(item.name)
    except Exception:
        controles = []
    return controles

def carregar_arquivos_ja_lidos(container_client, pasta_controle_base):
    """Carrega arquivos já lidos considerando todos os controles existentes da entidade."""
    arquivos_lidos = set()
    controles = listar_controles_entidade(container_client, pasta_controle_base)

    for caminho_controle in controles:
        registros = ler_json_adls(container_client, caminho_controle, padrao=[])
        for r in registros:
            arquivo = r.get("arquivo_origem")
            status = r.get("status")
            if arquivo and status == "LIDO":
                arquivos_lidos.add(arquivo)

    return arquivos_lidos, controles

def extrair_data_do_caminho_raw(caminho):
    """Extrai data do padrão real-time-data/YYYY/MM/DD/HHMMSS/arquivo.parquet."""
    try:
        partes = caminho.split("/")
        ano, mes, dia = int(partes[1]), int(partes[2]), int(partes[3])
        hora_min_seg = partes[4]
        hora = int(hora_min_seg[0:2])
        minuto = int(hora_min_seg[2:4])
        segundo = int(hora_min_seg[4:6])
        return datetime(ano, mes, dia, hora, minuto, segundo, tzinfo=timezone.utc)
    except Exception:
        return datetime.min.replace(tzinfo=timezone.utc)

In [0]:
# ==============================================================================
# CAMADA SILVER
# ==============================================================================
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

def ler_delta(camada: str, tabela: str, storage_opts: dict):
    """
    Lê uma tabela Delta física via SDK (bypass Serverless) 
    e converte para um DataFrame PySpark.
    """
    try:
        path = get_delta_path(camada, tabela, storage_opts)
        
        # Lê a tabela usando o SDK nativo em Rust
        dt = DeltaTable(path, storage_options=storage_opts)
        
        # Converte para Pandas e depois para Spark
        pdf = dt.to_pandas()
        return spark.createDataFrame(pdf)
        
    except Exception as e:
        print(f"[Erro] Falha interna ao tentar ler {camada}/{tabela}: {str(e)}")
        raise e




def schema_dq_logs():
    """
    Retorna o schema padronizado para a tabela de logs de qualidade de dados (DQ).
    """
    return StructType([
        StructField("run_id", StringType(), True),
        StructField("tabela", StringType(), True),
        StructField("regra", StringType(), True),
        StructField("status", StringType(), True),
        StructField("severidade", StringType(), True),
        StructField("qtd_registros_falhos", IntegerType(), True),
        StructField("qtd_registros_total", IntegerType(), True),
        StructField("timestamp_execucao", TimestampType(), True),
        StructField("arquivo_origem", StringType(), True)
    ])